# Design: W1994228146, Table 3 Equation (2)

## Source and target

The assigned scope is `table3_eq2_h1` / `table3_eq2_panel_a`. The target is Table 3 Equation (2), with the associated coefficient comparison and annual parameter tests in Panels B and C. The paper reports 30 separate annual cross-sectional regressions for fiscal years 1964–1993. It averages their coefficients and adjusted R-squared values; t-statistics use the mean and standard deviation of the 30 annual parameter estimates (physical PDF pp. 10–11; printed pp. 362–363). The fitted equation is `ROA_(t+1) = b0 + b1*GrLTNOA_t + b2*ACC_t + b3*ROA_t + error`.

## Data fields and temporal construction

The selected annual source is catalog-listed `COMP.FUNDA` (`/storage/sas/base/comp/d_na/funda.sas7bdat`), filtered to industrial, consolidated, domestic, standardized records (`INDFMT=INDL`, `CONSOL=C`, `POPSRC=D`, `DATAFMT=STD`). These key fields distinguish Compustat presentations; the Xpressfeed guide says multiple records for one company-period can represent different presentations. The merge key is GVKEY and fiscal year, with DATADATE retained and checked for chronological order. Ambiguous duplicate GVKEY-fiscal-year records are counted and excluded rather than resolved with an arbitrary first-row rule.

Current field mappings follow the paper's definitions (physical PDF pp. 5–6, 10–11) and the local Compustat field catalog: `OIADP` for operating income after depreciation and amortization (paper item 178), `AT` for total assets (item 6), `RECT` (item 2), `INVT` (item 3), `ACO` (item 68), `AP` (item 70), `LCO` (item 72), `DP` (item 14), `PPENT` (item 8), `INTAN` (item 33), `AO` (item 69), and `LO` (item 75). All current and lag/lead values are taken from same-GVKEY records at fiscal-year indices t−1, t, and t+1, with actual DATADATEs checked to increase. The complete supporting window is 1963–1994 for 1964–1993 regression anchors.

For t, average assets are `(AT_(t-1)+AT_t)/2`; `ROA_t = OIADP_t / average assets`. Operating accruals are `ACC_t = [(RECT_t-RECT_(t-1)) + (INVT_t-INVT_(t-1)) + (ACO_t-ACO_(t-1)) - (AP_t-AP_(t-1)) - (LCO_t-LCO_(t-1)) - DP_t] / average assets`. Net operating assets are `NOA = RECT + INVT + ACO + PPENT + INTAN + AO - AP - LCO - LO`; `GrNOA_t = (NOA_t-NOA_(t-1))/average assets`; `GrLTNOA_t = GrNOA_t-ACC_t`. The equivalent long-term change formula is `[(PPENT_t-PPENT_(t-1))+(INTAN_t-INTAN_(t-1))+(AO_t-AO_(t-1))-(LO_t-LO_(t-1))+DP_t]/average assets`; the direct and expanded formulas will be checked on complete observations. `ROA_(t+1)` uses `OIADP_(t+1)` over `(AT_t+AT_(t+1))/2`.

The Compustat monetary amounts are kept in their reported currency and scale. Adjacent-year balance-sheet differences and the t+1 outcome denominator require the same nonmissing `CURCD` across t−1, t, and t+1; no implicit FX conversion is applied. CRSP prices are screened separately and are not combined arithmetically with Compustat amounts.

The source says the final sample has available stock-price and financial-statement data in the prior, current, and subsequent years (physical PDF p. 5; printed p. 357). Read literally, that requires stock-price/share availability at all three annual anchors as well as financial data at t−1, t, and t+1. Attempt 2 already extended the price/share check from t to all three anchors. In this attempt, I repaired the CCM effective-date bounds in both the preliminary link filter and the anchor/monthly-price joins: `missing(linkdt)` and `missing(linkenddt)` preserve provider-defined open intervals, while ordinary finite bounds remain inclusive. The paper does not specify the exact monthly date rule; the implementation uses the latest CRSP month-end on or before each `DATADATE`, within the prior 31 days, under a primary `LINKTYPE` LC/LU and `LINKPRIM` P/C CCM link. Multiple qualifying securities still collapse to one firm-period presence flag. Price values and Compustat currency are not conflated.

WRDS CCM examples document `.B`/`.E` and missing start/end bounds as open intervals. SAS orders special missing values below dates, so a direct `LINKENDDT>=date` comparison rejected `.E`. The exact local audit found 92,711 CCM rows, 14,750 `.E` endpoints, and no missing/special start endpoints. Of P/C LC/LU links, 1,179 `.E` link rows overlap the 1963–1994 support window; this is a candidate-link count, not an affected-observation count. In the executed build, the CCM support set increased from 16,323 to 17,502 links and the firm-period CRSP presence set from 119,482 to 134,717 keys. The three-anchor presence count increased from 92,279 to 105,304; actual sample key changes are recorded in `review.md`.

The source says to remove (i) firm-years with Compustat notes indicating an unclassified balance sheet, (ii) voluntary accounting changes affecting operating income or working-capital accounts, and (iii) goodwill increases (physical PDF p. 5). The Compustat 2004 provider guide identifies code `GB` as an unclassified-balance-sheet presentation in which current-asset/current-liability classifications may be estimated (printed p. 86); the available `BSPR` field contains `GB`, so those observations are excluded. `ACCTCHG` is blank for the observed annual records, and the selected `FUNDA` file lacks the item-level accounting-change footnote data needed to isolate the source's voluntary changes; that screen remains unavailable and is not guessed. A detected goodwill increase is checked using the current semantic `GDWL` field. The source visibly prints “Compustat item 240” for goodwill, while the 2004 provider guide maps A240 to LIFR and defines GDWL separately; the exact historical crosswalk or source-number issue is unresolved, so the implementation uses GDWL by its documented semantic label and records this as a best-effort mapping.

All required components remain missing when any required source amount or temporal match is unavailable; no missing amount is filled with zero. The baseline sample is limited to 1964–1993 anchors with usable t−1, t, and t+1 records, positive average assets, valid currency continuity, CRSP price/share availability at all three anchors, no observed goodwill increase, and no documented `BSPR=GB` presentation. The paper's 32,961 count is a comparison result, not a target used to select records.

## Estimation and reported results

The notebook uses installed `xtfmb` with year as the time dimension and its default Fama–MacBeth covariance, which averages annual coefficients and uses their cross-year standard deviations while retaining covariance between coefficient series for the equality test. No serial-correlation correction is added because the paper describes t-statistics from the means and standard deviations of 30 annual estimates. Pooled OLS is not substituted. The paper does not report its within-year estimator or additional fixed effects, weights, or clusters, so ordinary unweighted annual OLS with an intercept is the documented best-effort choice. Panel A exports the average-coefficient regression table and average adjusted R-squared; Panel B tests equality of the two average coefficients; Panel C reports annual test counts at the source's two-sided 1% threshold. Annual group counts and failed/singular fits are logged. A descriptive table is produced for the actual analysis sample to support the Table 1 comparison. The repaired execution has 67,585 observations, all 30 annual fits estimable, and zero annual fit failures; estimates and paper comparisons are in `review.md`.

## Main unresolved items

The repaired sample is 67,585 versus 32,961 firm-years in the paper; the CCM correction adds 7,971 eligible keys to attempt 2, with no prior eligible key removed. The remaining difference is not attributed to a single cause. The local data are a current Compustat/CRSP vintage rather than the paper's Compustat 2000 / CRSP 2001 releases. Exact historical note-code filters and the old numeric goodwill crosswalk are unavailable in the selected catalog fields. The annual price/share date convention is not specified in the paper. These are recorded as fidelity limitations; none is used to tune the sample to the published N or coefficients.


In [ ]:
options errorabend;

libname comp "/storage/sas/base/comp/d_na" access=readonly;
libname crsp "/storage/sas/base/crsp/a_stock" access=readonly;
libname ccm "/storage/sas/base/crsp/a_ccm" access=readonly;
libname out "./data";

/* Keep the complete lag/lead support window before selecting regression years. */
data work.funda_raw;
    set comp.funda(
        keep=gvkey datadate fyear indfmt consol popsrc datafmt curcd
             oiadp at rect invt aco ap lco dp ppent intan ao lo gdwl
             acctchg bspr final fdate
        where=(fyear>=1963 and fyear<=1994 and indfmt='INDL'
               and consol='C' and popsrc='D' and datafmt='STD'));
    format datadate yymmdd10.;
run;

proc sql;
    create table work.funda_key_n as
    select gvkey, fyear, count(*) as nkey
    from work.funda_raw
    group by gvkey, fyear;

    create table work.funda_unique as
    select a.*
    from work.funda_raw as a
    inner join work.funda_key_n as k
      on a.gvkey=k.gvkey and a.fyear=k.fyear
    where k.nkey=1;

    select count(*) as funda_rows,
           count(distinct gvkey) as funda_gvkeys
    from work.funda_raw;
    select count(*) as ambiguous_firm_years,
           sum(nkey-1) as ambiguous_extra_rows
    from work.funda_key_n
    where nkey>1;
    select count(*) as unique_rows
    from work.funda_unique;
quit;

/* Documented BSPR=GB is the unclassified-balance-sheet code; ACCTCHG is unavailable here. */
proc freq data=work.funda_raw;
    tables bspr acctchg curcd final / missing;
run;

/* CRSP monthly price and shares for every fiscal-year point in the support window. */
data work.msf_support;
    set crsp.msf(keep=permno date prc shrout);
    where date>='01DEC1962'd and date<='31DEC1994'd;
run;

/* Primary effective links to actual CRSP securities across the support dates.
   CCM uses missing endpoints, including .B and .E, for open intervals. SAS
   sorts special missings below ordinary dates, so date bounds are explicit. */
data work.link_support;
    set ccm.ccmxpf_linktable(keep=gvkey lpermno linkdt linkenddt linkprim linktype);
    where linkprim in ('P','C') and linktype in ('LC','LU')
      and (missing(linkdt) or linkdt<='31DEC1994'd)
      and (missing(linkenddt) or linkenddt>='01JAN1963'd);
run;

/* Presence screen only; collapse multiple securities at each firm-period key. */
proc sql;
    create table work.price_key as
    select a.gvkey, a.fyear, a.datadate, max(1) as hasprice
    from work.funda_unique as a
    inner join work.link_support as l
      on a.gvkey=l.gvkey
     and (missing(l.linkdt) or l.linkdt<=a.datadate)
     and (missing(l.linkenddt) or l.linkenddt>=a.datadate)
    inner join work.msf_support as m
      on l.lpermno=m.permno
     and m.date<=a.datadate
     and m.date>=intnx('month',a.datadate,-1,'end')
     and (missing(l.linkdt) or m.date>=l.linkdt)
     and (missing(l.linkenddt) or m.date<=l.linkenddt)
     and not missing(m.prc) and not missing(m.shrout)
    where a.fyear between 1963 and 1994
    group by a.gvkey, a.fyear, a.datadate;

    create table work.fy_pairs as
    select a.*,
           p.datadate as datadate_l format=yymmdd10.,
           p.curcd as curcd_l,
           p.at as at_l,
           p.rect as rect_l, p.invt as invt_l, p.aco as aco_l,
           p.ap as ap_l, p.lco as lco_l,
           p.ppent as ppent_l, p.intan as intan_l,
           p.ao as ao_l, p.lo as lo_l, p.gdwl as gdwl_l,
           n.datadate as datadate_f format=yymmdd10.,
           n.curcd as curcd_f,
           n.oiadp as oiadp_f, n.at as at_f
    from work.funda_unique as a
    left join work.funda_unique as p
      on a.gvkey=p.gvkey and p.fyear=a.fyear-1 and p.datadate<a.datadate
    left join work.funda_unique as n
      on a.gvkey=n.gvkey and n.fyear=a.fyear+1 and n.datadate>a.datadate
    where a.fyear between 1964 and 1993;

    create table work.pairs_price as
    select a.*,
           coalesce(pl.hasprice,0) as hasprice_l,
           coalesce(pc.hasprice,0) as hasprice,
           coalesce(pf.hasprice,0) as hasprice_f
    from work.fy_pairs as a
    left join work.price_key as pl
      on a.gvkey=pl.gvkey and pl.fyear=a.fyear-1 and pl.datadate=a.datadate_l
    left join work.price_key as pc
      on a.gvkey=pc.gvkey and pc.fyear=a.fyear and pc.datadate=a.datadate
    left join work.price_key as pf
      on a.gvkey=pf.gvkey and pf.fyear=a.fyear+1 and pf.datadate=a.datadate_f;
quit;

data work.constructed;
    set work.pairs_price;
    length year 8;
    year=fyear;

    date_order=not missing(datadate_l) and not missing(datadate_f)
        and datadate_l<datadate and datadate_f>datadate;
    same_currency=not missing(curcd) and not missing(curcd_l) and not missing(curcd_f)
        and curcd_l=curcd and curcd_f=curcd;
    complete_inputs=(nmiss(oiadp,oiadp_f,at,at_l,at_f,
        rect,rect_l,invt,invt_l,aco,aco_l,ap,ap_l,lco,lco_l,dp,
        ppent,ppent_l,intan,intan_l,ao,ao_l,lo,lo_l)=0);

    goodwill_observed=(not missing(gdwl) and not missing(gdwl_l));
    goodwill_increase=(goodwill_observed and gdwl>gdwl_l);

    avgta=(at_l+at)/2;
    avgta_f=(at+at_f)/2;
    if complete_inputs and avgta>0 and avgta_f>0 then do;
        roa=oiadp/avgta;
        roa_f1=oiadp_f/avgta_f;
        grwc_raw=(rect-rect_l)+(invt-invt_l)+(aco-aco_l)
                 -(ap-ap_l)-(lco-lco_l);
        grwc=grwc_raw/avgta;
        depamort=-dp/avgta;
        acc=(grwc_raw-dp)/avgta;
        cfo=roa-acc;
        noa=rect+invt+aco+ppent+intan+ao-ap-lco-lo;
        noa_l=rect_l+invt_l+aco_l+ppent_l+intan_l+ao_l-ap_l-lco_l-lo_l;
        grnoa=(noa-noa_l)/avgta;
        grltnoa_full=grnoa-acc;
        grltnoa=((ppent-ppent_l)+(intan-intan_l)+(ao-ao_l)
                -(lo-lo_l)+dp)/avgta;
        grlt_check=grltnoa_full-grltnoa;
    end;
    else call missing(roa,roa_f1,grwc,depamort,acc,cfo,noa,noa_l,
                      grnoa,grltnoa_full,grltnoa,grlt_check);

    model_complete=complete_inputs and avgta>0 and avgta_f>0
        and not missing(roa) and not missing(roa_f1)
        and not missing(acc) and not missing(grltnoa)
        and not missing(grnoa);
    unclassified_bspr=(bspr='GB');
    prices_complete=(hasprice_l=1 and hasprice=1 and hasprice_f=1);
    baseline=(date_order and same_currency and prices_complete
              and model_complete and not goodwill_increase
              and not unclassified_bspr);
    label roa_f1='One-year-ahead ROA'
          roa='Current ROA'
          acc='Operating accruals'
          cfo='Cash flows from operations'
          grltnoa='Growth in long-term net operating assets'
          grnoa='Growth in net operating assets'
          grwc='Growth in working capital'
          depamort='Depreciation and amortization (negative)';
run;

proc sql;
    select count(*) as anchor_rows,
           sum(date_order=1) as with_ordered_lag_lead,
           sum(same_currency=1) as same_currency_rows,
           sum(hasprice_l=1) as with_lag_crsp_price_shares,
           sum(hasprice=1) as with_current_crsp_price_shares,
           sum(hasprice_f=1) as with_lead_crsp_price_shares,
           sum(prices_complete=1) as crsp_price_3period,
           sum(model_complete=1) as complete_model_rows,
           sum(unclassified_bspr=1) as unclassified_balance_sheets,
           sum(goodwill_increase=1) as observed_goodwill_increases,
           sum(goodwill_observed=0) as missing_goodwill_comparisons,
           sum(baseline=1) as final_baseline_rows
    from work.constructed;
    select year, count(*) as n
    from work.constructed
    where baseline=1
    group by year
    order by year;
    select count(*) as formula_check_n,
           max(abs(grlt_check)) as max_abs_formula_difference
    from work.constructed
    where model_complete=1;
quit;

proc means data=work.constructed n nmiss mean std p25 median p75 min max maxdec=6;
    where baseline=1;
    var roa_f1 roa acc cfo grltnoa grnoa grwc depamort;
run;

data out.main;
    set work.constructed;
    where baseline=1;
    keep gvkey year datadate curcd roa_f1 roa acc cfo grltnoa grnoa grwc depamort
         hasprice_l hasprice hasprice_f same_currency date_order goodwill_increase
         goodwill_observed bspr acctchg;
run;
